# Nền tảng 10 — Viết báo cáo thực nghiệm từ chính kết quả này

Chín notebook trước dựng nền lý thuyết. Notebook này là bước cuối: biến kết quả đã có thành một báo cáo mà người
đọc có thể **kiểm tra lại** và **không bị dẫn tới kết luận mạnh hơn dữ liệu cho phép**.

Ba phần:

1. **Cấu trúc** — mỗi mục của một bài thực nghiệm trả lời câu hỏi nào, và phần nào của project đã có sẵn nội dung.
2. **Con số** — bao nhiêu chữ số được phép viết, báo cáo hiệu ứng kèm gì, sinh thẳng các bảng từ file kết quả.
3. **Hiệu chỉnh phát biểu** — viết kết luận đúng mức, và viết mục hạn chế cho đàng hoàng. Đây là phần quyết định
   một báo cáo trung thực hay không.

Các cell trong notebook sinh **đúng** những bảng và hình sẽ đưa vào báo cáo, nên nó cũng là công cụ làm việc chứ
không chỉ để đọc.

In [ ]:
import json
import math
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
RUNS = ROOT / "kaggle" / "outputs"
KET_QUA = ROOT / "results"

from vitok import analysis
from vitok.stats import bpc, paired_bootstrap_bpc

runs = analysis.load(RUNS)
print(f"đã nạp {len(runs)} file kết quả:")
for (cond, depth, seed) in sorted(runs):
    print(f"  {cond:10s} d{depth:<3d} seed {seed}")

## 1. Cấu trúc một bài thực nghiệm

Mỗi mục tồn tại để trả lời **một** câu hỏi của người đọc. Nếu một mục không trả lời câu hỏi của nó, người đọc sẽ
dừng lại ở đó.

| Mục | Câu hỏi người đọc đang có | Nguồn trong project |
|---|---|---|
| Tóm tắt | "Bài này tìm ra cái gì?" | H1/H2/H3/H4 kèm con số |
| Giới thiệu | "Vì sao câu hỏi này đáng hỏi?" | tiếng Việt viết rời âm tiết → SuperBPE có cơ sở lý thuyết |
| Công trình liên quan | "Ai đã làm gì, và bài này khác chỗ nào?" | SuperBPE (2503.13423), MonTok (2510.21909) |
| Phương pháp | "Cụ thể đã làm gì?" | `vitok/train_tokenizers.py`, `vitok/superbpe.py`, notebook 04 |
| Thiết lập thí nghiệm | "Tôi lặp lại được không?" | `vitok/conditions.py`, `docs/analysis_plan.md`, notebook 03 |
| Kết quả | "Số liệu là gì?" | `results/summary.md` |
| Phân tích | "Vì sao lại ra như vậy?" | H4 wordhood, phân tích độ nhạy, notebook 05 |
| Hạn chế | "Tôi nên tin đến mức nào?" | mục 5 dưới đây |
| Kết luận | "Tôi mang gì đi?" | phát biểu đã hiệu chỉnh ở mục 6 |

Hai điểm dễ làm sai ở cấp cấu trúc:

**Phân cấp bằng chứng phải được tuyên bố, không phải suy ra.** `docs/analysis_plan.md` xếp H1 là chính, H2 là
phụ, H4 là khám phá, và được commit **trước khi train**. Trong báo cáo phải nói rõ điều đó, vì nó là thứ cho phép
người đọc tin rằng giả thuyết không được chọn sau khi nhìn dữ liệu (notebook 02 mục 12).

**Kết quả âm tính được báo cáo nguyên vẹn.** Project này có ít nhất ba kết quả âm tính hoặc đảo chiều, và chúng
là phần thú vị nhất của bài.

In [ ]:
print("các phát biểu đã đăng ký trước, đọc từ docs/analysis_plan.md:\n")
plan = (ROOT / "docs" / "analysis_plan.md").read_text(encoding="utf-8").splitlines()
trong_bang = False
for d in plan:
    if d.startswith("| # |"):
        trong_bang = True
    if trong_bang:
        if not d.startswith("|"):
            break
        print("  " + d[:150])

## 2. Bao nhiêu chữ số được phép viết

Một con số viết bốn chữ số thập phân **tuyên bố** rằng bốn chữ số đó có nghĩa. Quy tắc thực dụng: chữ số cuối
cùng được viết nên là chữ số đầu tiên bị nhiễu chạm tới, tức khoảng

$$\text{chữ số có nghĩa cuối} \approx \lfloor \log_{10} \mathrm{SE} \rfloor$$

**Ký hiệu mới:** $\lfloor \cdot \rfloor$ — làm tròn xuống. Ví dụ $\log_{10} 0{,}0003 \approx -3{,}5$,
$\lfloor -3{,}5 \rfloor = -4$: dừng ở chữ số thứ tư sau dấu phẩy.

Với project, hai nguồn nhiễu cho hai câu trả lời khác nhau, và đây là chỗ phải cẩn thận:

- Theo **bootstrap văn bản**: $\mathrm{SE} \approx 0{,}0003$ → chữ số thứ tư sau dấu phẩy là chữ số cuối còn nghĩa.
- Theo **nhiễu giữa hai lần train**: $\approx 0{,}0005$ → cũng dừng ở chữ số thứ tư, nhưng sát ngưỡng: chênh lệch
  $0{,}0002$ giữa hai con số bốn chữ số đã nằm gọn trong nhiễu train, nên không được diễn giải.

Nên viết `+0,0018` là hợp lý, còn `+0,00179` thì không. Và mọi con số bpc tuyệt đối (ví dụ `1,0019`) chỉ nên
giữ 4 chữ số thập phân, vì chữ số thứ năm nhỏ hơn nhiễu train hàng chục lần.

In [ ]:
def chu_so_hop_ly(se):
    return max(0, -int(math.floor(math.log10(se))))


for ten, se in (("bootstrap văn bản (d8, H1)", 0.000309), ("nhiễu giữa hai seed", 0.0005),
                ("nhiễu seed trên văn bản bỏ dấu", 0.007)):
    n = chu_so_hop_ly(se)
    print(f"  {ten:32s}: SE = {se:.6f} -> viết tối đa {n} chữ số thập phân"
          f"  (ví dụ {0.0017893625:.{n}f})")

print("\nmột ví dụ về cái sai hay gặp: chép nguyên giá trị float ra báo cáo")
d = 0.0017893625205913644
print(f"  nguyên bản     : {d}")
print(f"  hợp lý          : {d:+.4f}")
print(f"  kèm khoảng tin cậy: {d:+.4f} [{0.0011818941:+.4f}, {0.0023862618:+.4f}]")
print(f"  kèm thanh nhiễu seed: {d:+.4f} ± 0,0005 (giữa hai seed của cùng điều kiện)")

### Mỗi hiệu ứng phải đi kèm bốn thứ

1. **Độ lớn tuyệt đối** — $+0{,}0018$ bpc.
2. **Độ lớn tương đối** — $0{,}18\%$ so với $1{,}0019$ bpc của mốc so sánh.
3. **Khoảng bất định** — CI 95% từ bootstrap.
4. **Thanh nhiễu không nằm trong CI** — nhiễu giữa hai lần train.

Thiếu (2), người đọc không biết hiệu ứng có đáng kể về mặt thực tiễn không (notebook 02 mục 10). Thiếu (4), người
đọc sẽ tưởng CI đã bao phủ mọi nguồn ngẫu nhiên, mà nó thì không (notebook 02 mục 11).

In [ ]:
depth = 8
at = {k: v for k, v in runs.items() if k[1] == depth and k[2] == 0}
idx = analysis.common_docs(list(at.values()), "clean")
chars = np.array(at[("bpe-nfc", depth, 0)]["docs"]["clean"]["chars"])[idx]
na = analysis.arr(at[("super-nfc", depth, 0)], "clean", idx)
nb = analysis.arr(at[("bpe-nfc", depth, 0)], "clean", idx)
res = paired_bootstrap_bpc(na, nb, chars)
moc = bpc(nb, chars)

print("một dòng kết quả viết đủ bốn thành phần:\n")
print(f"  Δbpc(super-nfc − bpe-nfc) = {res['diff']:+.4f} bpc"
      f" ({res['diff'] / moc:+.2%} tương đối so với {moc:.4f}),")
print(f"  CI 95% [{res['ci95'][0]:+.4f}, {res['ci95'][1]:+.4f}] (bootstrap ghép cặp, B = 10.000, n = {len(idx)}),")
print(f"  so với chênh lệch giữa hai seed của cùng điều kiện: ±0,0005 bpc.")

## 3. Sinh bảng thẳng từ file kết quả

Chép số bằng tay là nguồn sai sót không cần thiết, và khiến báo cáo lệch khỏi dữ liệu mỗi lần chạy lại. Toàn bộ
bảng của project sinh từ `vitok.analysis`, và cell dưới in ra đúng nội dung sẽ dán vào báo cáo.

In [ ]:
compression = json.loads((RUNS / "vitok-data" / "compression-16k.json").read_text())
bang = analysis.summarize(runs, compression)
print(bang[:2600])

In [ ]:
print(analysis.sensitivity(runs))
print(analysis.scaling(runs))

Phân tích độ nhạy ở trên trả lời một câu hỏi mà người phản biện chắc chắn sẽ hỏi: **kết quả có do một nhóm nhỏ
văn bản kéo đi không?** Cách kiểm tra: bỏ 5% văn bản dài nhất, rồi chia đôi tập theo độ dài, và xem dấu của hiệu
ứng có đổi không. Mọi tập con vẫn giữ ghép cặp, nên phép so vẫn hợp lệ (notebook 02 mục 7.4).

Cột `docs favouring A` còn cho một cách nhìn khác: hiệu ứng trung bình $+0{,}0018$ không có nghĩa SuperBPE thua ở
mọi văn bản — ở d8 nó vẫn thắng ở 882/1996 văn bản. Con số này nên có trong báo cáo, vì nó mô tả **phân bố** chứ
không chỉ trung bình.

## 4. Hình vẽ

Một hình chỉ xứng đáng chiếm chỗ nếu nó nói được thứ mà bảng không nói. Với project, hình đó là H3: **dấu của
hiệu ứng đổi theo cỡ model** — một xu hướng, thứ mắt người đọc nhanh hơn đọc ba con số.

In [ ]:
fig = ROOT / "figures" / "h3_scaling.png"
analysis.scaling(runs, figures=ROOT / "figures")
print(f"hình đã sinh: {fig} ({fig.stat().st_size / 1024:.0f} KB)" if fig.exists() else "chưa có hình")

print("\nnguyên tắc cho hình trong báo cáo:")
for i, s in enumerate([
    "trục có nhãn và đơn vị (bpc, số tham số thân) — không ai đoán được 'y' là gì",
    "trục x là số tham số, thang log, vì các cỡ model cách nhau theo bội số",
    "vẽ thanh nhiễu seed nếu có; không có thì nói rõ trong chú thích",
    "chú thích hình phải đọc hiểu được khi tách rời khỏi bài",
    "không vẽ đường nối ngoại suy ra ngoài khoảng đã đo (notebook 03 mục 9)",
], 1):
    print(f"  {i}. {s}")

## 5. Mục hạn chế

Mục hạn chế viết tốt **tăng** độ tin cậy của bài, vì nó cho thấy tác giả biết chính xác kết quả của mình đứng
vững tới đâu. Viết vụng thì thành hai thái cực đều tệ: hoặc liệt kê chung chung ("cần thêm thí nghiệm"), hoặc
giấu hạn chế thật ở chỗ khó thấy.

Quy tắc: **mỗi hạn chế phải có hướng và độ lớn**. "Có thể ảnh hưởng tới kết quả" là vô dụng; "thiên lệch chống
lại SuperBPE khoảng 20% compute" là dùng được.

Bảy hạn chế của project, kèm số đo và hướng:

In [ ]:
han_che = [
    ("Compute không bằng nhau",
     "Thiết kế equal-text cho SuperBPE ít token hơn ~18%, nên ít FLOPs hơn ~21% cho cùng lượng văn bản.",
     "Bất lợi cho SuperBPE -> hiệu ứng 'SuperBPE kém hơn ở d8/d10' có thể bị thổi lên.",
     "notebook 03 mục 7"),
    ("Chỉ hai seed",
     "Chênh lệch giữa hai seed của cùng điều kiện ở d8: <=0,0005 bpc (clean), 0,0015-0,0095 (văn bản bỏ dấu); mỗi con số là MỘT hiệu số.",
     "Không rõ hướng; mọi kết luận H2 ở d8 nằm trong vùng này nên không đứng vững.",
     "notebook 02 mục 11"),
    ("Seed chỉ đổi khởi tạo",
     "NANOCHAT_SEED tác động vào torch.manual_seed; thứ tự dữ liệu giữ nguyên giữa các seed.",
     "Thanh nhiễu đo được là CẬN DƯỚI của nhiễu thật.",
     "notebook 02 mục 11.3"),
    ("Cặp tối thiểu chạm trần",
     "Độ chính xác 0,992-0,996 -> chỉ 11-20 cặp bất đồng -> power gần 0.",
     "Không kết luận được gì từ phép đo này, KHÔNG phải 'hai model như nhau'.",
     "notebook 02 mục 9, notebook 05 mục 7"),
    ("Chính tả không chuẩn hoá",
     "'hòa' và 'hoà' là hai dãy code point khác nhau và không được hợp nhất.",
     "Chia đôi tần suất của cùng một từ; ảnh hưởng mọi điều kiện như nhau nên ít khả năng đổi kết luận.",
     "notebook 05 mục 3"),
    ("Một corpus, một ngôn ngữ",
     "Tokenizer huấn luyện trên 500MB FineWeb-2 vie_Latn; văn bản web, không có văn bản chuyên ngành.",
     "Không rõ hướng; kết luận chỉ áp cho miền văn bản này.",
     "docs/analysis_plan.md mục 2"),
    ("Ba điểm trên trục cỡ model",
     "d6/d8/d10 trải 4,6 lần về số tham số thân; mọi 'xu hướng' chỉ là nội suy trong khoảng đó.",
     "Không cho phép phát biểu về model lớn hơn.",
     "notebook 03 mục 10"),
]

for ten, do, huong, xem in han_che:
    print(f"▸ {ten}")
    print(f"    số đo  : {do}")
    print(f"    hướng  : {huong}")
    print(f"    chi tiết: {xem}\n")

## 6. Hiệu chỉnh phát biểu

Đây là kỹ năng khó nhất của việc viết kết quả: nói đúng những gì dữ liệu cho phép, không hơn, không kém. Ba lỗi
thường gặp, kèm bản sửa dùng chính số liệu của project.

| Phát biểu quá tay | Vì sao sai | Bản đã hiệu chỉnh |
|---|---|---|
| "SuperBPE làm model tiếng Việt kém đi." | Dấu đổi theo cỡ model; ở d6 SuperBPE **tốt hơn**. | "Ở d8 và d10, SuperBPE cho bpc cao hơn BPE $0{,}18\%$ và $0{,}30\%$; ở d6 thì thấp hơn $0{,}16\%$." |
| "NFD giúp model bền hơn với văn bản thiếu dấu." | Chỉ đúng ở d6 và chỉ với `strip50`; ở `strip100` thì ngược lại. | "Ở d6, NFD tốt hơn NFC $0{,}0115$ bpc trên văn bản bỏ dấu 50%, nhưng kém hơn $0{,}0171$ bpc khi bỏ dấu hoàn toàn; ở d8 mọi hiệu ứng nằm trong nhiễu giữa hai lần train." |
| "Hai model tương đương trên bộ cặp tối thiểu ($p = 0{,}55$)." | $p$ lớn vì power gần 0, không phải vì bằng nhau. | "Bộ cặp tối thiểu không phân biệt được các điều kiện: độ chính xác 0,99+ để lại 11 cặp bất đồng, mức mà phép kiểm định gần như không có power." |

Hai mẫu câu dùng được cho gần như mọi kết quả:

- Khi có hiệu ứng: *"[hiệu ứng] là [số] ([số] tương đối), CI 95% [khoảng], so với thanh nhiễu giữa hai lần train
  là [số]."*
- Khi không có: *"Không phát hiện được khác biệt; với thiết kế này, hiệu ứng nhỏ nhất phát hiện được là [MDE], nên
  kết quả tương thích với mọi hiệu ứng thật nhỏ hơn mức đó."*

Mẫu thứ hai là cách viết kết quả âm tính cho đàng hoàng: nó biến "chúng tôi không thấy gì" thành một phát biểu có
nội dung, bằng cách nói rõ độ nhạy của phép đo.

In [ ]:
z95, z80 = 1.959964, 0.841621
se = paired_bootstrap_bpc(na, nb, chars)["ci95"]
se = (se[1] - se[0]) / (2 * z95)
mde = (z95 + z80) * se
print("câu mẫu cho kết quả âm tính, điền số thật của project (d8, clean):\n")
print(f'  "Không phát hiện được khác biệt vượt mức nhiễu; với n = {len(idx)} văn bản và thiết kế ghép cặp,')
print(f'   hiệu ứng nhỏ nhất phát hiện được ở power 80% là {mde:.4f} bpc ({mde / moc:.2%} tương đối),')
print(f'   nên kết quả tương thích với mọi hiệu ứng thật nhỏ hơn mức đó."')

print("\ncâu mẫu cho kết quả dương tính:\n")
print(f'  "SuperBPE cho bpc cao hơn BPE {res["diff"]:+.4f} ({res["diff"] / moc:+.2%} tương đối),')
print(f'   CI 95% [{res["ci95"][0]:+.4f}, {res["ci95"][1]:+.4f}], so với thanh nhiễu giữa hai lần train ±0,0005."')

## 7. Danh sách kiểm trước khi nộp

Tái lập được — người khác chạy lại có ra cùng số không:

- [ ] Commit hash của `nanochat` và bản vá (`patches/NANOCHAT_COMMIT`, `patches/nanochat.patch`).
- [ ] Phiên bản thư viện (`pixi.lock`), phần cứng (Kaggle T4×2), dtype (`float16`).
- [ ] Seed và điều gì seed **không** kiểm soát (thứ tự dữ liệu).
- [ ] Lệnh chính xác đã chạy, gồm mọi cờ (`vitok.conditions` sinh ra chúng).
- [ ] Cách lấy dữ liệu và cách tách test (lọc trùng theo hash).

Trung thực về thống kê:

- [ ] `docs/analysis_plan.md` được commit trước kết quả đầu tiên, và báo cáo nói rõ điều đó.
- [ ] Mọi so sánh đã chạy đều được báo cáo, kể cả kết quả âm tính.
- [ ] Bài toán so sánh nhiều lần được nêu (7 so sánh mỗi cỡ model, FWER 30% nếu coi tất cả ngang nhau).
- [ ] Mọi hiệu ứng có cả độ lớn tuyệt đối, tương đối, CI, và đối chiếu với nhiễu seed.
- [ ] Không có chữ số nào vượt quá mức nhiễu cho phép.

Nội dung:

- [ ] Kết luận nằm trong khoảng dữ liệu đã đo (không ngoại suy sang model lớn hơn).
- [ ] Mục hạn chế nêu đủ bảy điểm ở mục 5, mỗi điểm có hướng và độ lớn.
- [ ] Mọi bảng và hình sinh từ script, không chép tay.
- [ ] Mỗi hình có chú thích tự đứng được.

## 8. Câu hỏi tự kiểm

1. Vì sao phải nói rõ trong bài rằng kế hoạch phân tích được commit trước khi train?
2. $\mathrm{SE} = 0{,}0003$ và nhiễu seed $0{,}0005$. Viết Δbpc với bao nhiêu chữ số thập phân?
3. Một hiệu ứng có $p = 10^{-8}$ nhưng độ lớn $0{,}02\%$ tương đối. Viết một câu mô tả đúng cả hai mặt.
4. Vì sao "cần thêm thí nghiệm" không phải một hạn chế hợp lệ?
5. Cột `docs favouring A` bổ sung thông tin gì mà Δbpc trung bình không có?
6. Viết lại cho đúng mức: "Kết quả cho thấy NFD không có tác dụng ở d8."
7. Vì sao phân tích độ nhạy (bỏ 5% văn bản dài nhất) vẫn giữ được tính hợp lệ của phép so ghép cặp?
8. Báo cáo nên nói gì về chênh lệch compute giữa SuperBPE và BPE?

**Đáp án gợi ý**

1. Vì nó loại trừ khả năng giả thuyết được chọn sau khi nhìn dữ liệu; không có nó, người đọc phải giả định mức
   xấu nhất về bậc tự do của người nghiên cứu.
2. Bốn chữ số, và nên kèm thanh nhiễu ±0,0005 để người đọc biết chữ số thứ tư đã sát mức nhiễu.
3. "Hiệu ứng rất chắc chắn khác 0 ($p = 10^{-8}$) nhưng rất nhỏ về mặt thực tiễn (0,02% bpc), nên không đủ để
   thay đổi lựa chọn tokenizer."
4. Vì nó không nói người đọc nên giảm niềm tin vào **điều gì** và **bao nhiêu**; mọi bài đều cần thêm thí nghiệm.
5. Phân bố: hiệu ứng trung bình nhỏ có thể là "thua đều ở mọi văn bản" hoặc "thắng ở phần lớn nhưng thua đậm ở
   một số" — hai tình huống rất khác nhau.
6. "Ở d8, các hiệu ứng của NFD trên văn bản bỏ dấu (0,0007 và 0,0051 bpc) không vượt chênh lệch giữa hai lần train
   của cùng một điều kiện trên các biến thể đó (0,0015–0,0095 bpc), nên thí nghiệm này không kết luận được về NFD
   ở cỡ model đó."
7. Vì mọi tập con vẫn được áp cho **cả hai** điều kiện cùng lúc, nên từng cặp $(a_i, b_i)$ vẫn nguyên vẹn.
8. Nêu rõ hướng và độ lớn: equal-text cho SuperBPE ít hơn ~21% FLOPs, tức thiên lệch **chống lại** SuperBPE, nên
   phần hiệu ứng bất lợi quan sát được có thể một phần do compute chứ không phải do tokenizer.

**Nguồn đọc thêm**

- [Simon Peyton Jones, How to Write a Great Research Paper](https://www.microsoft.com/en-us/research/academic-program/write-great-research-paper/).
- [Workshop on Insights from Negative Results in NLP](https://aclanthology.org/venues/insights/) — mẫu viết kết quả âm tính.
- [Beyond Accuracy: Behavioral Testing with CheckList](https://aclanthology.org/2020.acl-main.442/) — mục 2, ý tưởng cặp tối thiểu.
- `docs/analysis_plan.md` và `results/summary.md` — nội dung sẵn có để dựng báo cáo.